In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import statsmodels.formula.api as smf

from scipy.stats import false_discovery_control

from cat_whim.config import PROCESSED_DATA_DIR

2024-09-18 14:28:47.502 | INFO     | cat_whim.config:<module>:11 - PROJ_ROOT path is: /home/leoner/Projects/cat_whim


In [16]:
# Change to df_cat_whim_to_analyze_complete_apoe.csv in case you want only ptids with complete info
df_filename = PROCESSED_DATA_DIR / "df_cat_whim_to_analyze.csv"
df_m00_filename = PROCESSED_DATA_DIR / "df_cat_whim_to_analyze_m00.csv"
df_long_mean_filename = PROCESSED_DATA_DIR / "df_cat_whim_to_analyze_long_format_mean.csv"
df_long_region_filename = PROCESSED_DATA_DIR / "df_cat_whim_to_analyze_long_format_regional.csv"
dk_regions_filename = PROCESSED_DATA_DIR / "dk_region_names.csv"

# Load data
df = pd.read_csv(df_filename)
df = df[df["session"]!="M02"].reset_index().copy()
df_m00 = pd.read_csv(df_m00_filename)
df_long_mean = pd.read_csv(df_long_mean_filename)
df_long_region = pd.read_csv(df_long_region_filename)
dk_region_labels = pd.read_csv(dk_regions_filename).values.flatten()

# Get the region names for each biomarker
amy_cols = [region + "_SUVR_amy" for region in dk_region_labels]
tau_cols = [region + "_SUVR_tau" for region in dk_region_labels]
disconn_cols = [region + "_disconn" for region in dk_region_labels]
thick_cols = [region + "_THICKNESS" for region in dk_region_labels]

In [17]:
new_region_names = []

dict_values = {}

for region_name in dk_region_labels[:34]:

    region_names = region_name.split("_")

    new_region_name = region_names[0] + "_" + region_names[2]
    
    left_name = region_names[0] + "_LH_" + region_names[2] 
    right_name = region_names[0] + "_RH_" + region_names[2]

    thickness_lh = left_name + "_THICKNESS"
    amyloid_lh = left_name + "_SUVR_amy"
    tau_lh = left_name + "_SUVR_tau"
    disconn_lh = left_name + "_disconn"

    thickness_rh = right_name + "_THICKNESS"
    amyloid_rh = right_name + "_SUVR_amy"
    tau_rh = right_name + "_SUVR_tau"
    disconn_rh = right_name + "_disconn"

    dict_values[f"{new_region_name}_thickness"] = (df[thickness_lh].values + df[thickness_rh].values) / 2
    dict_values[f"{new_region_name}_SUVR_amy"] = (df[amyloid_lh].values + df[amyloid_rh].values) / 2
    dict_values[f"{new_region_name}_SUVR_tau"] = (df[tau_lh].values + df[tau_rh].values) / 2
    dict_values[f"{new_region_name}_disconn"] = (df[disconn_lh].values + df[disconn_rh].values) / 2

    new_region_names.append(new_region_name)

df_avg = pd.DataFrame(dict_values)

In [18]:
df_new = pd.concat([df, df_avg], axis=1)

(850, 432)

In [44]:
df_new_m00 = df_new[df_new["session"] == "M00"]

In [53]:
list_to_include= []
for col in dk_region_labels:
    disconn_col = col + "_disconn"
    v = (df_new_m00[disconn_col] == 0).mean()
    if v < 0.75:
        list_to_include.append(col)

In [55]:
from tqdm import tqdm
import numpy as np

list_coeff_disconn = []
list_p_values_disconn = []

list_coeff_disconn_years = []
list_p_values_disconn_years = []


for region in tqdm(list_to_include):

    thickness = region + "_THICKNESS"
    disconn = region + "_disconn"
    
    formula = f"""{thickness} ~ {disconn}*Years_m00 + C(PTGENDER) + AGE + C(HMHYPERT) + CTX_ETIV"""
    
    lmm = smf.mixedlm(data=df_new, formula=formula, groups=df_new["PTID"])
    res = lmm.fit()    

    coeff_disconn = res.params[disconn]
    p_values_disconn = res.pvalues[disconn]

    coeff_disconn_years = res.params[f"{disconn}:Years_m00"]
    p_values_disconn_years = res.pvalues[f"{disconn}:Years_m00"]

    list_coeff_disconn.append(coeff_disconn_years)
    list_p_values_disconn.append(p_values_disconn)

    list_coeff_disconn_years.append(coeff_disconn_years)
    list_p_values_disconn_years.append(p_values_disconn_years)


 25%|██▌       | 13/51 [00:09<00:28,  1.34it/s]/home/leoner/miniconda3/envs/cat_whim/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
 31%|███▏      | 16/51 [00:11<00:25,  1.36it/s]/home/leoner/miniconda3/envs/cat_whim/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
100%|██████████| 51/51 [00:37<00:00,  1.36it/s]


In [57]:
arr_p_values_disconn = np.array(list_p_values_disconn)
arr_p_values_disconn_years = np.array(list_p_values_disconn_years)

arr_p_values_disconn_fdr = false_discovery_control(arr_p_values_disconn)
arr_p_values_disconn_years_fdr = false_discovery_control(arr_p_values_disconn_years)

print("Disconn regions:", np.array(list_to_include)[arr_p_values_disconn_fdr < 0.05])
print("Disconn/amy rate regions:", np.array(list_to_include)[arr_p_values_disconn_fdr < 0.05])

Disconn regions: ['CTX_RH_FUSIFORM' 'CTX_RH_INSULA']
Disconn/amy rate regions: ['CTX_RH_FUSIFORM' 'CTX_RH_INSULA']


In [ ]:
from tqdm import tqdm
import numpy as np

list_coeff_amy = []
list_p_values_amy = []

list_coeff_tau = []
list_p_values_tau = []

list_coeff_disconn = []
list_p_values_disconn = []

list_coeff_amy_years = []
list_p_values_amy_years = []

list_coeff_tau_years = []
list_p_values_tau_years = []

list_coeff_disconn_years = []
list_p_values_disconn_years = []

list_has_converged = []

for region in tqdm(new_region_name):

    thickness = region + "_THICKNESS"
    amyloid = region + "_SUVR_amy"
    tau = region + "_SUVR_tau"
    disconn = region + "_disconn"

    # formula = f"{thickness} ~ {amyloid} + {tau} + {disconn} + C(PTGENDER) + AGE + C(HMHYPERT) + C(TRACER_amy) + CTX_ETIV + C(APOE4)"
    # ols = smf.ols(data=dfino, formula=formula)
    # res =  ols.fit()
    try:
        formula = f"""{disconn}*Years_m00 + 
                      C(PTGENDER) + AGE + C(HMHYPERT) + CTX_ETIV"""
        
        lmm = smf.mixedlm(data=df, formula=formula, groups=df["PTID"])
        res = lmm.fit()    

        coeff_amy = res.params[amyloid]
        p_values_amy = res.pvalues[amyloid]

        coeff_amy_years = res.params[f"{amyloid}:Years_m00"]
        p_values_amy_years = res.pvalues[f"{amyloid}:Years_m00"]

        coeff_tau = res.params[tau]
        p_values_tau = res.pvalues[tau]

        coeff_tau_years = res.params[f"{tau}:Years_m00"]
        p_values_tau_years = res.pvalues[f"{tau}:Years_m00"]

        coeff_disconn = res.params[disconn]
        p_values_disconn = res.pvalues[disconn]

        coeff_disconn_years = res.params[f"{disconn}:Years_m00"]
        p_values_disconn_years = res.pvalues[f"{disconn}:Years_m00"]

        list_coeff_amy.append(coeff_amy)
        list_p_values_amy.append(p_values_amy)

        list_coeff_tau.append(coeff_tau)
        list_p_values_tau.append(p_values_tau)

        list_coeff_disconn.append(coeff_disconn)
        list_p_values_disconn.append(p_values_disconn)

        list_coeff_amy_years.append(coeff_amy_years)
        list_p_values_amy_years.append(p_values_amy_years)

        list_coeff_tau_years.append(coeff_tau_years)
        list_p_values_tau_years.append(p_values_tau_years)

        list_coeff_disconn_years.append(coeff_disconn_years)
        list_p_values_disconn_years.append(p_values_disconn_years)

        list_has_converged.append(res.converged)

    except:
        list_coeff_amy.append(np.nan)
        list_p_values_amy.append(1)

        list_coeff_tau.append(np.nan)
        list_p_values_tau.append(1)

        list_coeff_disconn.append(np.nan)
        list_p_values_disconn.append(1)

        list_coeff_amy_years.append(np.nan)
        list_p_values_amy_years.append(1)

        list_coeff_tau_years.append(np.nan)
        list_p_values_tau_years.append(1)

        list_coeff_disconn_years.append(np.nan)
        list_p_values_disconn_years.append(1)

        list_has_converged.append(False)

In [14]:
arr_p_values_amy = np.array(list_p_values_amy)
arr_p_values_tau = np.array(list_p_values_tau)
arr_p_values_disconn = np.array(list_p_values_disconn)

arr_p_values_amy_years = np.array(list_p_values_amy_years)
arr_p_values_tau_years = np.array(list_p_values_tau_years)
arr_p_values_disconn_years = np.array(list_p_values_disconn_years)

arr_p_values_amy_fdr = false_discovery_control(arr_p_values_amy)
arr_p_values_tau_fdr = false_discovery_control(arr_p_values_tau)
arr_p_values_disconn_fdr = false_discovery_control(arr_p_values_disconn)

arr_p_values_amy_years_fdr = false_discovery_control(arr_p_values_amy_years)
arr_p_values_tau_years_fdr = false_discovery_control(arr_p_values_tau_years)
arr_p_values_disconn_years_fdr = false_discovery_control(arr_p_values_disconn_years)

print("Amyloid regions:", dk_region_labels[arr_p_values_amy_fdr < 0.05])
print("Tau regions:", dk_region_labels[arr_p_values_tau_fdr < 0.05])
print("Disconn regions:", dk_region_labels[arr_p_values_disconn_fdr < 0.05])

print("Amy/tau rate regions:", dk_region_labels[arr_p_values_amy_fdr < 0.05])
print("Tau/disconn rate regions:", dk_region_labels[arr_p_values_tau_fdr < 0.05])
print("Disconn/amy rate regions:", dk_region_labels[arr_p_values_disconn_fdr < 0.05])